In [1]:
import pandas as pd
import joblib
import numpy as np

### wczytywanie obiektu
df_with_all_clusters = joblib.load('df_with_all_clusters.joblib')

##### podgląd próbki
print("\nPodgląd próbki DataFrame:")
print(df_with_all_clusters.head())


Podgląd próbki DataFrame:
                                               title  \
0  Clinical features of culture-proven Mycoplasma...   
1  Nitric oxide: a pro-inflammatory mediator in l...   
2    Surfactant protein-D and pulmonary host defense   
3               Role of endothelin-1 in lung disease   
4  Gene expression in epithelial cells in respons...   

                                            abstract  \
0  OBJECTIVE: This retrospective chart review des...   
1  Inflammatory diseases of the respiratory tract...   
2  Surfactant protein-D (SP-D) participates in th...   
3  Endothelin-1 (ET-1) is a 21 amino acid peptide...   
4  Respiratory syncytial virus (RSV) and pneumoni...   

                                                text  \
0  Clinical features of culture-proven Mycoplasma...   
1  Nitric oxide: a pro-inflammatory mediator in l...   
2  Surfactant protein-D and pulmonary host defens...   
3  Role of endothelin-1 in lung disease Endotheli...   
4  Gene expression 

In [3]:
import sqlite3
import time

### Create SQLite Database from DataFrame ---

#### database and table names
db_filename = 'covid19_analysis.db'
table_name = 'articles'

print(f"Przygotowanie do zapisu DataFrame do pliku bazy danych SQLite: {db_filename}, tabela: {table_name}")
start_time = time.time()

#### connection to the SQLite database
conn = sqlite3.connect(db_filename)

#### save the DataFrame to the SQL table
#    - 'conn': the database connection object
#    - 'name': the name of the table to create/overwrite
#    - 'if_exists='replace'': if the table already exists, drop it and create a new one
#    - 'index=False': do not write the DataFrame index as a column in the SQL table
#    - 'chunksize=10000': write data in batches of 10000 rows for memory efficiency
df_with_all_clusters.to_sql(name=table_name, con=conn, if_exists='replace', index=False, chunksize=10000)

#### close the database connection
conn.close()

end_time = time.time()
print(f"DataFrame zapisany do bazy danych SQLite w {(end_time - start_time):.2f} sekundy.")
print(f"Utworzono plik bazy danych SQLite: {db_filename}")

Przygotowanie do zapisu DataFrame do pliku bazy danych SQLite: covid19_analysis.db, tabela: articles
DataFrame zapisany do bazy danych SQLite w 14.46 sekundy.
Utworzono plik bazy danych SQLite: covid19_analysis.db


In [1]:
import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb
import sqlite3
import time

### wczytywanie obiektów
df = joblib.load('df_with_all_clusters.joblib')
model = joblib.load('lgb_gbdt_best_model_ver4.joblib')
X = np.load('umap_embeddings_10d.npy')
print("Obiekty wczytane.")

Obiekty wczytane.


In [3]:
### Generowanie predykcji dla całego zbioru danych

print("\nGenerowanie predykcji za pomocą najlepszego modelu LightGBM...")
start_time = time.time()
predictions_full = model.predict(X)
end_time = time.time()
print(f"Predykcje wygenerowane w {(end_time - start_time):.2f} sekund.")


Generowanie predykcji za pomocą najlepszego modelu LightGBM...


O:\Projects\fashion_mnist\pyth311env\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Predykcje wygenerowane w 411.73 sekund.


In [5]:
### dodanie Predykcji jako nowej kolumny
new_column_name = 'predicted_topic_lgbm_gbdt_v4'
df[new_column_name] = predictions_full
print(f"Dodano nową kolumnę: '{new_column_name}'")
print("\nPodgląd DataFrame z nową kolumną:")
print(df[['nmf_topic', new_column_name]].head())

Dodano nową kolumnę: 'predicted_topic_lgbm_gbdt_v4'

Podgląd DataFrame z nową kolumną:
   nmf_topic  predicted_topic_lgbm_gbdt_v4
0          0                             0
1          9                             9
2          8                             8
3          9                             9
4          8                             8


In [7]:
### aktualizacja plików
print("\nAktualizowanie pliku DataFrame i bazy danych SQLite...")

### zapis / aktualizacja DataFrame
start_time = time.time()
joblib.dump(df, 'df_with_all_clusters.joblib')
print(f"- DataFrame zaktualizowany w {(time.time() - start_time):.2f} sek.")


Aktualizowanie pliku DataFrame i bazy danych SQLite...
- DataFrame zaktualizowany w 2.82 sek.


In [9]:
### zapis zaktualizowanego DataFrame do bazy SQLite (nadpisanie tabeli)
db_filename = 'covid19_analysis.db'
table_name = 'articles'
start_time = time.time()
conn = sqlite3.connect(db_filename)
df.to_sql(name=table_name, con=conn, if_exists='replace', index=False, chunksize=10000)
conn.close()
print(f"- Baza danych SQLite zaktualizowana w {(time.time() - start_time):.2f} sek.")

- Baza danych SQLite zaktualizowana w 75.63 sek.
